[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_answering.ipynb)

# DataSage Stage 3: Answering GRPO Training

Pre-builds dataset with real environment observations (no `rollout_func`).
Reward functions replay the env with stored seeds for context-matched evaluation.
Includes Patronus Lynx integration for hallucination detection (optional).

**Requirements:** GPU (A100/H100), `HF_TOKEN`, `WANDB_API_KEY`.  
**Optional:** `PATRONUS_API_KEY` for Patronus Lynx.

In [ ]:
!pip install -q unsloth trl datasets wandb pydantic requests
!pip install -q openenv-core
!pip install -q patronus  # optional

In [ ]:
import os, sys

REPO_URL = "https://github.com/ricalanis/openenv-hackaton.git"  # UPDATE THIS
REPO_DIR = "/content/datasage"

if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Load API keys from Colab Secrets or set manually
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    # Optional: for Patronus Lynx hallucination detection
    os.environ["PATRONUS_API_KEY"] = userdata.get("PATRONUS_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."
# os.environ["PATRONUS_API_KEY"] = "pat_..."  # optional

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

In [ ]:
# Imports
import random
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from training.shared.config import BASE_MODEL, HF_MODEL_REPOS, SPACE_URLS, TRAINING_CONFIGS, WANDB_PROJECT
from training.shared.parsers import parse_answering_action
from training.shared.rewards import (
    patronus_reward_fn, answering_json_format_reward, persona_match_reward)
from environments.answering.client import AnsweringEnv
from environments.answering.models import AnsweringAction

ENV_URL = SPACE_URLS["answering"]
STAGE_CONFIG = TRAINING_CONFIGS["answering"]
print(f"Environment URL: {ENV_URL}"); print(f"Config: {STAGE_CONFIG}")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded: {BASE_MODEL}")

In [ ]:
SYSTEM_PROMPT = """\
You are a data-driven enterprise analyst. You answer questions about \
datasets across multiple domains (HR, Sales, Project Management, \
IT Operations) tailored to the audience persona.

Personas:
- Executive: Focus on costs, ROI, strategic risk, portfolio trends, year-over-year metrics.
- Manager: Focus on team performance, operational health, process bottlenecks, capacity.
- Individual Contributor: Focus on personal tasks, deadlines, what to do next.

Respond with a JSON answer:
{"answer": "<your answer>", "cited_columns": ["col1", "col2"], "reasoning": "<chain-of-thought>"}

Rules:
1. Ground every claim in the data.
2. Match your tone and vocabulary to the persona.
3. Be concise but thorough.
4. Never fabricate numbers."""

TASK_DESCRIPTIONS = [
    "Answer the question based on the data, tailored to the persona.",
    "Provide a data-grounded answer appropriate for this audience.",
    "Analyze the data and answer the question in the persona's style.",
    "Use the dataset to answer accurately, matching the persona's focus.",
    "Generate a faithful, persona-aligned answer citing real data.",
    "Answer using statistics from the data, in the right tone for this persona.",
    "Review the data summary and answer the question for this stakeholder.",
    "Craft a response grounded in the data that matches the persona's needs.",
]

In [ ]:
# Dataset: pre-built with real environment observations
random.seed(42)
SEEDS = [42 + i for i in range(64)]
prompts = []
persona_names = []

print("Building dataset with real environment observations...")
for i, seed in enumerate(SEEDS):
    task_desc = random.choice(TASK_DESCRIPTIONS)
    with AnsweringEnv(base_url=ENV_URL) as client:
        result = client.reset_with_seed(seed=seed)
        obs = result.observation

    prompts.append([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Domain: {obs.domain}\nPersona: {obs.persona}\n"
            f"Persona Focus: {obs.persona_description}\n\n"
            f"Question: {obs.question}\n\n"
            f"Dataset Summary:\n{obs.dataset_summary}\n\n"
            f"Column Statistics:\n{obs.column_stats}\n\n"
            f"Available Columns: {', '.join(obs.available_columns)}\n\n"
            f"Task: {task_desc}"
        )},
    ])
    persona_names.append(obs.persona)
    if (i + 1) % 16 == 0:
        print(f"  Built {i + 1}/64 examples")

dataset = Dataset.from_dict({
    "prompt": prompts,
    "seed": SEEDS,
    "persona_name": persona_names,
})
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Environment reward function (calls env directly with stored seeds)
def answering_env_reward(completions: list[str], **kwargs) -> list[float]:
    """Primary reward: replay env with stored seed, step with parsed action."""
    seeds = kwargs.get("seed", [0] * len(completions))
    rewards = []
    for text, seed in zip(completions, seeds):
        try:
            action_dict = parse_answering_action(text)
            action = AnsweringAction(
                answer=action_dict.get("answer", text),
                cited_columns=action_dict.get("cited_columns", []),
                reasoning=action_dict.get("reasoning", ""),
            )
            with AnsweringEnv(base_url=ENV_URL) as client:
                client.reset_with_seed(seed=int(seed))
                result = client.step(action)
                rewards.append(float(result.reward or 0.0))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

In [ ]:
training_args = GRPOConfig(
    output_dir="./outputs/answering-grpo",
    learning_rate=STAGE_CONFIG["learning_rate"], num_train_epochs=STAGE_CONFIG["num_train_epochs"],
    per_device_train_batch_size=STAGE_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=STAGE_CONFIG["gradient_accumulation_steps"],
    num_generations=STAGE_CONFIG["num_generations"],
    max_completion_length=STAGE_CONFIG["max_completion_length"],
    max_prompt_length=STAGE_CONFIG["max_prompt_length"],
    logging_steps=1, save_steps=50, bf16=True,
    use_vllm=True, vllm_mode="colocate",
    report_to="wandb", run_name="datasage-answering-grpo-v2")
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

In [ ]:
# Create trainer and train
trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args,
    train_dataset=dataset,
    reward_funcs=[answering_env_reward, patronus_reward_fn, answering_json_format_reward, persona_match_reward],
)
print("Starting Stage 3 (Answering) GRPO training...")
trainer.train()

In [ ]:
# Save and push to Hub
output_dir = "./outputs/answering-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

hf_repo = HF_MODEL_REPOS["answering"]
trainer.push_to_hub(hf_repo)
print(f"Model pushed to https://huggingface.co/{hf_repo}")